<a href="https://colab.research.google.com/github/Nanda-Lopes/AlgoStudies/blob/main/Stanford_Algorithms_Specialization_Course4_W4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Problema 2-SAT via Componentes Fortemente Conexas (SCC)

O problema de Satisfatibilidade Booleana com cláusulas de tamanho 2 (2-SAT) consiste em determinar se existe uma atribuição de valores de verdade (Verdadeiro ou Falso) para as variáveis booleanas que satisfaça simultaneamente todas as cláusulas da fórmula na Forma Normal Conjuntiva (FNC).

Embora o problema 3-SAT seja NP-completo, o problema 2-SAT admite resolução exata em tempo linear $O(V + E)$ por meio da redução para o cálculo de Componentes Fortemente Conexas (SCC) em um Grafo de Implicações:

1. **Construção do Grafo de Implicações:**
   - Para cada variável booleana $x_i$, dois vértices foram definidos no grafo: um para a variável positiva ($x_i$) e outro para sua negação ($\neg x_i$).
   - Cada cláusula disjuntiva $(u \lor v)$ foi convertida em duas implicações lógicas equivalentes:
$$(\neg u \implies v) \quad \text{e} \quad (\neg v \implies u)$$
   - Duas arestas direcionadas foram adicionadas ao grafo: de $\neg u$ para $v$, e de $\neg v$ para $u$.

2. **Critério de Satisfatibilidade (Teorema de Aspvall, Plass e Tarjan):**
   - Uma fórmula 2-SAT é insatisfatível se, e somente se, existir alguma variável $x_i$ tal que $x_i$ e sua negação $\neg x_i$ pertençam à mesma Componente Fortemente Conexa (SCC).
   - A existência desse ciclo mútuo de implicações significaria que $x_i \implies \dots \implies \neg x_i$ e $\neg x_i \implies \dots \implies x_i$, gerando uma contradição lógica insolúvel.
   - Caso contrário, a fórmula é garantidamente satisfatível, e uma valoração consistente pode ser obtida por meio da ordenação topológica inversa das componentes.

O algoritmo de **Tarjan** para identificação de SCCs foi implementado de forma linear sobre o grafo direcionado de implicações.

In [1]:
import os
import sys

sys.setrecursionlimit(2000000)

def load_2sat_instance(filename):
    if not os.path.exists(filename) and os.path.exists(filename + ".txt"):
        filename = filename + ".txt"
    elif not os.path.exists(filename) and os.path.exists(filename.replace(".txt", "")):
        filename = filename.replace(".txt", "")

    with open(filename, "r") as f:
        first_line = f.readline()
        num_vars = int(first_line.strip())
        adj = [[] for _ in range(2 * num_vars + 1)]
        for line in f:
            if line.strip():
                u, v = map(int, line.split())
                def var_to_idx(lit):
                    if lit > 0:
                        return lit
                    else:
                        return num_vars + abs(lit)
                def neg_idx(lit):
                    if lit > 0:
                        return num_vars + lit
                    else:
                        return abs(lit)
                adj[neg_idx(u)].append(var_to_idx(v))
                adj[neg_idx(v)].append(var_to_idx(u))
    return num_vars, adj

def solve_2sat(filename):
    num_vars, adj = load_2sat_instance(filename)
    total_nodes = 2 * num_vars

    indices = [-1] * (total_nodes + 1)
    lowlink = [-1] * (total_nodes + 1)
    on_stack = [False] * (total_nodes + 1)
    stack = []
    scc_id = [-1] * (total_nodes + 1)

    timer = 0
    current_scc = 0

    for u in range(1, total_nodes + 1):
        if indices[u] == -1:
            call_stack = [(u, 0)]
            while call_stack:
                node, edge_idx = call_stack[-1]
                if edge_idx == 0:
                    indices[node] = timer
                    lowlink[node] = timer
                    timer += 1
                    stack.append(node)
                    on_stack[node] = True

                if edge_idx < len(adj[node]):
                    neighbor = adj[node][edge_idx]
                    call_stack[-1] = (node, edge_idx + 1)
                    if indices[neighbor] == -1:
                        call_stack.append((neighbor, 0))
                    elif on_stack[neighbor]:
                        if indices[neighbor] < lowlink[node]:
                            lowlink[node] = indices[neighbor]
                else:
                    call_stack.pop()
                    if call_stack:
                        parent, _ = call_stack[-1]
                        if lowlink[node] < lowlink[parent]:
                            lowlink[parent] = lowlink[node]

                    if lowlink[node] == indices[node]:
                        while True:
                            w = stack.pop()
                            on_stack[w] = False
                            scc_id[w] = current_scc
                            if w == node:
                                break
                        current_scc += 1

    for x in range(1, num_vars + 1):
        pos_node = x
        neg_node = num_vars + x
        if scc_id[pos_node] == scc_id[neg_node]:
            return 0
    return 1

# Execução das 6 Instâncias e Determinação da Satisfatibilidade

As 6 instâncias de teste (`2sat1`, `2sat2`, `2sat3`, `2sat4`, `2sat5` e `2sat6`) foram processadas sequencialmente.

Para cada instância, se a fórmula for satisfatível, o bit correspondente é definido como `1`; caso contenha contradições de ciclo implicacional, o bit é definido como `0`. A resposta final corresponde à concatenação desses 6 valores em uma única cadeia de bits.

In [2]:
instances = ["2sat1", "2sat2", "2sat3", "2sat4", "2sat5", "2sat6"]
results = []

for inst in instances:
    res = solve_2sat(inst)
    results.append(str(res))
    status = "SATISFIABLE" if res == 1 else "UNSATISFIABLE"
    print(f"Instância {inst}: {status} ({res})")

final_bitstring = "".join(results)
print("==================================")
print("RESPOSTA FINAL (6-BIT STRING):")
print(final_bitstring)
print("==================================")

Instância 2sat1: SATISFIABLE (1)
Instância 2sat2: UNSATISFIABLE (0)
Instância 2sat3: SATISFIABLE (1)
Instância 2sat4: SATISFIABLE (1)
Instância 2sat5: UNSATISFIABLE (0)
Instância 2sat6: UNSATISFIABLE (0)
RESPOSTA FINAL (6-BIT STRING):
101100
